In [71]:
import pandas as pd
import optuna
# optuna.logging.set_verbosity(optuna.logging.ERROR)

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.base import clone
from sklearn.metrics import make_scorer

In [39]:
# Ссылка на raw-версию файла из репозитория GitHub
final_data = "https://raw.githubusercontent.com/Khamoon7/GeoATM-popularity/refs/heads/main/data/train_data.csv"

# Чтение файла
df = pd.read_csv(final_data)

# Отображение данных
display(df)

,id,atm_group,address_raw,address_geocoded,geo_lon,geo_lat,country,region,municipality,city,...,nearest_public_transport_dist_m,count_public_transport_300m,nearest_parking_dist_m,count_parking_300m,nearest_education_dist_m,count_education_300m,nearest_subway_dist_m,nearest_post_offices_dist_m,count_post_offices_300m,has_subway_nearby
0,5.0,496.5,BUDENNOGO 7A ELISTA,"Россия, Республика Калмыкия, Элиста, улица С.М...",44.260605,46.318231,Россия,Республика Калмыкия,городской округ Элиста,Элиста,...,93.6,5,143.7,3,247.3,1,0.0,0.0,0,False
1,6.0,496.5,"HO CHI MIHN AVE, 19 ULYANOVSK","Россия, Ульяновск, проспект Хо Ши Мина, 19",48.300652,54.270443,Россия,Ульяновская область,городской округ Ульяновск,Ульяновск,...,89.9,6,NaN,0,260.8,1,0.0,220.4,2,False
2,7.0,496.5,SHELESTA 116A KHABAROVSK,"Россия, Хабаровск, улица Шелеста, 116А",135.052594,48.520497,Россия,Хабаровский край,городской округ Хабаровск,Хабаровск,...,33.8,8,112.9,6,0.0,0,0.0,186.6,2,False
3,8.0,496.5,ORDZHONIKIDZE 52 YAKUTSK,"Россия, Республика Саха (Якутия), Якутск, улиц...",129.721308,62.025566,Россия,Республика Саха (Якутия),городской округ Якутск,Якутск,...,119.8,7,246.9,4,195.4,5,0.0,167.2,1,False
4,10.0,496.5,"VETERANOV AVE, 3 KRASNOKAMENS","Россия, Забайкальский край, Краснокаменск, про...",118.027480,50.090714,Россия,Забайкальский край,Краснокаменский муниципальный округ,Краснокаменск,...,70.6,3,48.3,5,NaN,0,NaN,NaN,0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6195,8806.0,496.5,CHKALOVA 14 S. TROITSKOY,"Россия, Республика Калмыкия, Целинный район, с...",44.253338,46.417932,Россия,Республика Калмыкия,Целинный район,село Троицкое,...,192.4,1,631.5,0,NaN,0,NaN,NaN,0,False
6196,8807.0,496.5,OKTYABRSKAYA 18A IKI-BURUL,"Россия, Республика Калмыкия, посёлок Ики-Бурул...",44.647851,45.826910,Россия,Республика Калмыкия,Ики-Бурульский район,посёлок Ики-Бурул,...,NaN,0,780.6,0,NaN,0,NaN,NaN,0,False
6197,8809.0,496.5,"ZHIGULSKOGO 1""Z"" LAGAN","Россия, Республика Калмыкия, Лагань, улица Жиг...",47.361688,45.389283,Россия,Республика Калмыкия,Лаганское городское муниципальное образование,Лагань,...,NaN,0,NaN,0,NaN,0,NaN,NaN,0,False
6198,8810.0,496.5,GORODOVIKOVA 3A ELISTA,"Россия, Республика Калмыкия, Элиста, улица Б. ...",44.266498,46.306946,Россия,Республика Калмыкия,городской округ Элиста,Элиста,...,131.8,2,144.1,11,112.2,2,0.0,0.0,0,False


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6200 entries, 0 to 6199
Data columns (total 48 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   id                                   6200 non-null   float64
 1   atm_group                            6200 non-null   float64
 2   address_raw                          6200 non-null   object 
 3   address_geocoded                     6200 non-null   object 
 4   geo_lon                              6200 non-null   float64
 5   geo_lat                              6200 non-null   float64
 6   country                              6200 non-null   object 
 7   region                               6200 non-null   object 
 8   municipality                         6200 non-null   object 
 9   city                                 6200 non-null   object 
 10  street                               6200 non-null   object 
 11  house                         

In [77]:
# Преобразуем булевы колонки в int (0/1)
df = df.apply(lambda c: c.astype(int) if pd.api.types.is_bool_dtype(c) else c)

# Колонки, которые нужно исключить из обучения
cols_to_drop = ["target", "id", "atm_group", "country", "address_raw", "address_geocoded"]

# Разделяем данные на признаки и целевую переменную
y = df["target"]
X = df.drop(columns=cols_to_drop) 

# Определяем числовые и категориальные признаки
num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

# Препроцессинг: медианный импутер для числовых и частотный + OneHotEncoder для категориальных
preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ],
    remainder="drop"
)

# Базовая модель дерева решений
model = DecisionTreeRegressor(random_state=42, max_depth=12, min_samples_leaf=20, min_samples_split=40)

# Общий пайплайн: препроцессинг + модель
pipe = Pipeline([("prep", preprocess), ("models", model)])

# Делим данные на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

# Настройки кросс-валидации и метрики
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(r2_score)

# Целевая функция для Optuna (оптимизация R²)
def objective(trial):
    # Поиск оптимальных гиперпараметров DecisionTreeRegressor
    params = {
        "model__max_depth": trial.suggest_int("model__max_depth", 4, 60),
        "model__min_samples_leaf": trial.suggest_int("model__min_samples_leaf", 1, 300),
        "model__min_samples_split": trial.suggest_int("model__min_samples_split", 2, 400),
        "model__max_features": trial.suggest_categorical("model__max_features", [None, "sqrt", "log2"]),
        "model__splitter": trial.suggest_categorical("model__splitter", ["best", "random"]),
    }

    # Клонируем пайплайн и применяем параметры
    pipe_trial = clone(pipe)
    pipe_trial.set_params(**params)

    # Считаем средний R² по фолдам
    scores = cross_val_score(pipe_trial, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1)
    return scores.mean()


# Настраиваем и запускаем исследование Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=80, show_progress_bar=False)

# Выводим лучшие параметры и метрику по CV
print("Лучший средний R² (CV)", study.best_value)
print("Лучшие параметры модели:", study.best_params)

# Обучаем модель с лучшими параметрами на тренировочных данных
best_pipe = clone(pipe).set_params(**study.best_params)
best_pipe.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred = best_pipe.predict(X_test)

# Финальные метрики на тесте
print("\nИтоговые метрики на тесте")
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))
print("RMSE:", round(root_mean_squared_error(y_test, y_pred), 4))
print("R2:", round(r2_score(y_test, y_pred), 4))

# Важность признаков
model_ = best_pipe.named_steps["models"]
feat_names = best_pipe.named_steps["prep"].get_feature_names_out()
fi = pd.Series(model_.feature_importances_, index=feat_names).sort_values(ascending=False)
fi_df = fi.reset_index()
fi_df.columns = ["Признак", "Важность"]
fi_df = fi_df.sort_values("Важность", ascending=False)

# Топ-10 наиболее значимых признаков
display(fi_df.head(10).style.set_caption("Топ признаков по важности"))

Лучший средний R² (CV) 0.43016545215279606
Лучшие параметры модели: {'model__max_depth': 13, 'model__min_samples_leaf': 11, 'model__min_samples_split': 288, 'model__max_features': None, 'model__splitter': 'random'}

Итоговые метрики на тесте
MAE: 0.0501
RMSE: 0.0675
R2: 0.4045


,Признак,Важность
0,num__cash_out,0.524055
1,cat__region_Республика Татарстан (Татарстан),0.101637
2,num__is_24_7,0.084616
3,cat__region_Москва,0.045375
4,num__cash_in,0.039182
5,cat__municipality_городской округ Кызыл,0.028417
6,num__account_statement,0.026651
7,num__loan_payments,0.026463
8,num__geo_lon,0.016020
9,cat__region_Приморский край,0.014071


## Отчёт по модели DecisionTreeRegressor (Optuna + OHE)

### Данные и препроцессинг
- Булевы признаки → `int` (0/1).
- Разделение: `train_test_split(test_size=0.25, random_state=42)`.
- Препроцессинг:
  - Числовые: `SimpleImputer(strategy="median")`
  - Категориальные: `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore")`.

### Алгоритм
- Модель: `DecisionTreeRegressor`
- Поиск гиперпараметров: **Optuna**, 80 испытаний, CV=5 (R²).

### Лучшие параметры (Optuna)
```python
{
    "model__max_depth": 13,
    "model__min_samples_leaf": 11,
    "model__min_samples_split": 288,
    "model__max_features": None,
    "model__splitter": "random"
}
```

### Качество модели
**Лучший средний R² (CV):** `0.4302`  

**Метрики на тесте:**
- **MAE:** `0.0501`
- **RMSE:** `0.0675`
- **R²:** `0.4045`

---

### Топ-10 признаков по важности (после препроцессинга)
| №  | Признак | Важность |
|----|----------|-----------|
| 1  | num__cash_out | 0.5241 |
| 2  | cat__region_Республика Татарстан (Татарстан) | 0.1016 |
| 3  | num__is_24_7 | 0.0846 |
| 4  | cat__region_Москва | 0.0454 |
| 5  | num__cash_in | 0.0392 |
| 6  | cat__municipality_городской округ Кызыл | 0.0284 |
| 7  | num__account_statement | 0.0267 |
| 8  | num__loan_payments | 0.0265 |
| 9  | num__geo_lon | 0.0160 |
| 10 | cat__region_Приморский край | 0.0141 |


### Краткие выводы

- Единичное дерево с `OneHotEncoding` демонстрирует стабильный уровень качества — **R² ~0.39–0.43**, что является **нормальным базовым результатом**.
- Наибольший вклад в предсказания вносят **функциональные флаги** (`cash_out`, `is_24_7`, `cash_in`, `account_statement`) и **географический сигнал** через `region`, `municipality`, `geo_lon`.
- Параметры с сильной регуляризацией (`min_samples_leaf`, `min_samples_split`, `splitter='random'`) уменьшают переобучение и повышают устойчивость модели.

---

### Рекомендации по улучшению

1. **Проверить ансамблевые модели:**
   - `RandomForestRegressor`
   - `CatBoostRegressor`  

2. **Улучшить геопризнаки:**
   - заменить `region/city` на **геоячейки** по координатам (`geo_lat`, `geo_lon`),  
     округлив до сетки (например, 0.05°) → более устойчивая фича.

3. **Усилить валидацию:**
   - использовать `GroupKFold` по `region` или `city`,  
     если важна **пространственная устойчивость** модели.

Далее идет проверка RF

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline([
    ("prep", preprocess),
    ("models", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    ))
])

# Кросс-валидация и метрика
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(r2_score)


rf.fit(X_train, y_train)
yp = rf.predict(X_test)

print("\n[RF] Метрики на тесте")
print("MAE:", round(mean_absolute_error(y_test, yp), 4))
print("RMSE:", round(root_mean_squared_error(y_test, yp), 4))
print("R2:", round(r2_score(y_test, yp), 4))

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf_pipe = Pipeline([
    ("prep", preprocess),
    ("models", RandomForestRegressor(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        n_jobs=-1,
        random_state=42
    ))
])

# Кросс-валидация и метрика
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scorer = make_scorer(r2_score)

criterion_choices = ["squared_error", "friedman_mse"]  # при желании можно добавить "absolute_error" (иногда медленнее)

# Целевая функция для Optuna
def rf_objective(trial):
    # Комментарий: max_features — либо доля [0.2–1.0], либо из готовых вариантов
    use_fraction = trial.suggest_categorical("use_fraction_max_features", [True, False])
    if use_fraction:
        max_features_val = trial.suggest_float("model__max_features_frac", 0.2, 1.0)
    else:
        max_features_val = trial.suggest_categorical("model__max_features_cat", [None, "sqrt", "log2"])

    # Комментарий: бутстрэп + (опционально) подвыборка объектов для деревьев
    bootstrap_val = trial.suggest_categorical("model__bootstrap", [True, False])
    if bootstrap_val:
        max_samples_val = trial.suggest_float("model__max_samples", 0.5, 1.0)
    else:
        max_samples_val = None

    params = {
        # базовые
        "model__n_estimators": trial.suggest_int("model__n_estimators", 300, 1200),
        "model__criterion": trial.suggest_categorical("model__criterion", criterion_choices),
        "model__max_depth": trial.suggest_categorical("model__max_depth", [None] + list(range(5, 51))),
        "model__min_samples_leaf": trial.suggest_int("model__min_samples_leaf", 1, 80),
        "model__min_samples_split": trial.suggest_int("model__min_samples_split", 2, 200),
        "model__max_features": max_features_val,
        "model__bootstrap": bootstrap_val,
        # регуляризация
        "model__min_impurity_decrease": trial.suggest_float("model__min_impurity_decrease", 1e-7, 1e-3, log=True),
        # подвыборка объектов, если bootstrap включен
        "model__max_samples": max_samples_val,
        # прочее
        "model__n_jobs": -1,
        "model__random_state": 42,
    }

    pipe_trial = clone(rf_pipe)
    pipe_trial.set_params(**params)
    scores = cross_val_score(pipe_trial, X_train, y_train, cv=cv, scoring=scorer, n_jobs=-1)
    return scores.mean()

# Запуск Optuna
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(rf_objective, n_trials=120, show_progress_bar=True)

print("Лучший средний R² (CV):", round(study_rf.best_value, 4))
print("Лучшие параметры модели:", study_rf.best_params)

  0%|          | 0/120 [00:00<?, ?it/s]

Лучший средний R² (CV): 0.5193
Лучшие параметры модели: {'use_fraction_max_features': True, 'model__max_features_frac': 0.39073217360144474, 'model__bootstrap': True, 'model__max_samples': 0.7433691372778383, 'model__n_estimators': 746, 'model__criterion': 'friedman_mse', 'model__max_depth': 28, 'model__min_samples_leaf': 2, 'model__min_samples_split': 6, 'model__min_impurity_decrease': 0.0007467433568205243}


ValueError: Invalid parameter 'use_fraction_max_features' for estimator Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['geo_lon', 'geo_lat',
                                                   'population_density_per_km2',
                                                   'is_24_7',
                                                   'contactless_tech',
                                                   'qr_codes', 'usd_available',
                                                   'eur_available', 'cash_in',
                                                   'cash_out', 'cashless_pay',
                                                   'account_statement',
                                                   'access_for_disabled',
                                                   'transfer_p2p',
                        

In [ ]:
best_params_raw = study_rf.best_params

best_params = {}
for k, v in best_params_raw.items():
    if k in ("use_fraction_max_features", "model__max_features_frac", "model__max_features_cat"):
        continue
    if k.startswith("model__"):
        best_params[k] = v

if best_params_raw.get("use_fraction_max_features", False):
    best_params["model__max_features"] = best_params_raw.get("model__max_features_frac")
else:
    best_params["model__max_features"] = best_params_raw.get("model__max_features_cat", None)

if not best_params.get("model__bootstrap", True):
    best_params["model__max_samples"] = None
else:
    if "model__max_samples" in best_params_raw:
        best_params["model__max_samples"] = best_params_raw["model__max_samples"]

# (опционально) отфильтруем только допустимые ключи для пайплайна
allowed = set(rf_pipe.get_params().keys())
best_params = {k: v for k, v in best_params.items() if k in allowed}

# Применяем параметры без повторного поиска
best_rf_pipe = clone(rf_pipe).set_params(**best_params)
best_rf_pipe.fit(X_train, y_train)
yp = best_rf_pipe.predict(X_test)

print("\n[RF] Метрики на тесте (best_rf_pipe)")
print("MAE:", round(mean_absolute_error(y_test, yp), 4))
print("RMSE:", round(root_mean_squared_error(y_test, yp), 4))
print("R2:", round(r2_score(y_test, yp), 4))


[RF] Метрики на тесте (best_rf_pipe)
MAE: 0.0452
RMSE: 0.0612
R2: 0.5103


In [ ]:
import json

# Сохраняем в JSON-файл
with open("../models/best_rf_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, ensure_ascii=False, indent=4)